# Train a Minilm-based Matrix Factorization router (RouteLLM `mf`)

This notebook re-trains the RouteLLM **matrix factorization** (`mf`) router using a **local embedding model (MiniLM)** instead of OpenAI's `text-embedding-3-small`. Everything that needs real compute runs here on the Colab GPU:

1. **Load & preprocess** the LMSYS Arena battle data (`lmsys/lmsys-arena-human-preference-55k`).
2. **Embed every prompt** with MiniLM (this is the largest cost, and GPU makes it fast).
3. **Train the tiny MF model** (only ~57K trainable params; the heavy lifting was upstream in the embedder).
4. **Save a checkpoint + embeddings + data** into a `colab_outputs/` folder that you download **once**.

After downloading, you'll wire the local `routellm` code to use MiniLM at inference instead of OpenAI (a small edit to `routellm/routers/matrix_factorization/model.py` — see the comments in this notebook's `build_inference_checkpoint` step and the separate local-side notes).

## 0. Config

Edit these to control **time budget** and **quality**.

| Tuning knob | Effect |
|---|---|
| `EMBEDDER` | Which local embedder. MiniLM-L6 = fastest/smallest. `bge-base-en-v1.5` or `bge-m3` = better quality, a bit slower. |
| `MAX_PROMPTS` | Cap how many prompts you embed. `None` = full 55k. Smaller = much faster. |
| `DIM` | Latent dim of the `mf` model (the `P`/`text_proj` size). Keep 128 to match the original. |
| `NUM_EPOCHS` | Training epochs. The model is tiny; you'll plateau early. Watch `best_test_acc`. |
| `BATCH_SIZE` | Larger = faster training. |

In [ ]:
# ====== CONFIG ======
EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"  # 384-dim
# Better quality options (higher-dim), set TEXT_DIM to match:
# EMBEDDER = "BAAI/bge-base-en-v1.5"  # 768-dim -> TEXT_DIM = 768
# EMBEDDER = "BAAI/bge-m3"           # 1024-dim -> TEXT_DIM = 1024

TEXT_DIM = 384          # embedding dim of the chosen EMBEDDER
DIM = 128               # latent dim of the mf model (matches original)
NUM_MODELS = 64         # number of model slots in MODEL_IDS (matches original)
NUM_EPOCHS = 60
BATCH_SIZE = 256
LR = 3e-4
WEIGHT_DECAY = 1e-5
ALPHA = 0.1             # prompt-embedding noise (regularization) during training
MAX_PROMPTS = None      # None = full 55k; e.g. 10000 for a quick run
MIN_LEN = 16            # drop prompts shorter than this (matches original preprocessing)

# Data source (public HuggingFace dataset, no API key needed)
BATTLES_DATASET = "lmsys/lmsys-arena-human-preference-55k"

OUT_DIR = "/content/colab_outputs"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print("Output directory:", OUT_DIR)

## 1. Install dependencies

In [ ]:
!pip -q install datasets sentence-transformers torch huggingface_hub numpy openai tqdm
print("installed")

## 2. Load and preprocess Arena battle data

Mirrors `routellm/routers/similarity_weighted/utils.py::preprocess_battles` + the `first_turn` extraction, so the row **order** of this filtered dataframe is what the `idx` values and the embedding array both index into.

In [ ]:
import json
import numpy as np
import pandas as pd
from datasets import load_dataset

print("Loading battles dataset...")
df = load_dataset(BATTLES_DATASET, split="train").to_pandas()
print("Raw rows:", len(df))

def get_first_turn(prompt_str):
    return json.loads(prompt_str)[0].strip()

def get_winner(row):
    if row["winner_model_a"] == 1:
        return "model_a"
    elif row["winner_model_b"] == 1:
        return "model_b"
    else:
        return "tie"

df["first_turn"] = df["prompt"].apply(get_first_turn)
df["winner"] = df.apply(get_winner, axis=1)

# Drop short prompts (same as original MIN_LEN filter) and keep needed cols in row order.
# We PRESERVE the integer row index so it can double as the embedding-table index (idx).
df = df.loc[df["first_turn"].apply(len) >= MIN_LEN]
df = df.reset_index(drop=True).reset_index()  # ... but keep a stable ordered index
df = df[["index", "model_a", "model_b", "winner", "first_turn"]]
df.columns = ["idx", "model_a", "model_b", "winner", "first_turn"]

# Optional cap for a quick run (embedding time + training cost)
if MAX_PROMPTS is not None:
    df = df.iloc[:MAX_PROMPTS].reset_index(drop=True)

print("Preprocessed rows:", len(df))
print(df.head())

## 3. Embed every prompt with MiniLM (GPU)

Aligns with the dataframe **row order** (`df["idx"]`), so `embeddings[i]` is the embedding of the prompt whose `idx == i`. Saved as `prompt_embeddings.npy` (shape `[num_prompts, TEXT_DIM]`).

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

print(f"Loading embedder {EMBEDDER}...")
embedder = SentenceTransformer(EMBEDDER, device=device)

prompts = df["first_turn"].tolist()
print(f"Embedding {len(prompts)} prompts...")
emb = embedder.encode(prompts, batch_size=128, show_progress_bar=True, convert_to_numpy=True)
print("Embedding shape:", emb.shape)
assert emb.shape == (len(prompts), TEXT_DIM), f"Unexpected shape {emb.shape}; check TEXT_DIM matches {EMBEDDER}"

np.save(os.path.join(OUT_DIR, "prompt_embeddings.npy"), emb)
print("Saved prompt_embeddings.npy")

## 4. Build pairwise training data

Filters battles to only decisive (`model_a`/`model_b` winner) **non-tied same-model** pairs, mirroring the main block of `train_matrix_factorization.py`. `idx` points at the matching row in `prompt_embeddings.npy`.

In [ ]:
pairwise = df[df["winner"].isin(["model_a", "model_b"]) & (df["model_a"] != df["model_b"])]
data = pairwise[["model_a", "model_b", "winner", "idx"]].to_dict(orient="records")

with open(os.path.join(OUT_DIR, "pairwise_data.json"), "w") as f:
    json.dump(data, f)
print("Pairwise samples:", len(data))

# Sanity: idx values must be in-bounds of the embedding array
max_idx = max(d["idx"] for d in data)
print("Max idx:", max_idx, "/ embeddings rows:", len(emb))
assert max_idx < len(emb), "idx out of range vs embeddings"
print(data[:3])

## 5. Training model (the tiny MF net)

Reimplements `MFModel_Train` from `routellm/routers/matrix_factorization/train_matrix_factorization.py` with `text_dim=TEXT_DIM` (our embedder) instead of 1536. Only `P`, `text_proj`, `classifier` are trainable; the `Q` prompt-embedding table is frozen. Adds **early stopping** on validation accuracy.

In [ ]:
import random
import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42); np.random.seed(42); random.seed(42)

# Clone of the MODEL_IDS map so training uses the same 64 model slots as inference.
# (Exact ids must match routellm/routers/matrix_factorization/model.py::MODEL_IDS)
MODEL_IDS = {
    "RWKV-4-Raven-14B": 0, "alpaca-13b": 1, "chatglm-6b": 2, "chatglm2-6b": 3,
    "chatglm3-6b": 4, "claude-1": 5, "claude-2.0": 6, "claude-2.1": 7,
    "claude-instant-1": 8, "codellama-34b-instruct": 9, "deepseek-llm-67b-chat": 10,
    "dolly-v2-12b": 11, "dolphin-2.2.1-mistral-7b": 12, "falcon-180b-chat": 13,
    "fastchat-t5-3b": 14, "gemini-pro": 15, "gemini-pro-dev-api": 16,
    "gpt-3.5-turbo-0125": 17, "gpt-3.5-turbo-0314": 18, "gpt-3.5-turbo-0613": 19,
    "gpt-3.5-turbo-1106": 20, "gpt-4-0125-preview": 21, "gpt-4-0314": 22,
    "gpt-4-0613": 23, "gpt-4-1106-preview": 24, "gpt4all-13b-snoozy": 25,
    "guanaco-33b": 26, "koala-13b": 27, "llama-13b": 28, "llama-2-13b-chat": 29,
    "llama-2-70b-chat": 30, "llama-2-7b-chat": 31, "llama2-70b-steerlm-chat": 32,
    "mistral-7b-instruct": 33, "mistral-7b-instruct-v0.2": 34, "mistral-medium": 35,
    "mixtral-8x7b-instruct-v0.1": 36, "mpt-30b-chat": 37, "mpt-7b-chat": 38,
    "nous-hermes-2-mixtral-8x7b-dpo": 39, "oasst-pythia-12b": 40, "openchat-3.5": 41,
    "openchat-3.5-0106": 42, "openhermes-2.5-mistral-7b": 43, "palm-2": 44,
    "pplx-70b-online": 45, "pplx-7b-online": 46, "qwen-14b-chat": 47,
    "qwen1.5-4b-chat": 48, "qwen1.5-72b-chat": 49, "qwen1.5-7b-chat": 50,
    "solar-10.7b-instruct-v1.0": 51, "stablelm-tuned-alpha-7b": 52,
    "starling-lm-7b-alpha": 53, "stripedhyena-nous-7b": 54, "tulu-2-dpo-70b": 55,
    "vicuna-13b": 56, "vicuna-33b": 57, "vicuna-7b": 58, "wizardlm-13b": 59,
    "wizardlm-70b": 60, "yi-34b-chat": 61, "zephyr-7b-alpha": 62, "zephyr-7b-beta": 63,
}
id2name = {v: k for k, v in MODEL_IDS.items()}


class PairwiseDataset(Dataset):
    def __init__(self, data):
        self.models_a = torch.tensor([MODEL_IDS[sample["model_a"]] for sample in data], dtype=torch.int64)
        self.models_b = torch.tensor([MODEL_IDS[sample["model_b"]] for sample in data], dtype=torch.int64)
        self.prompt_id = torch.tensor([sample["idx"] for sample in data], dtype=torch.int64)
        self.winners = [sample["winner"] for sample in data]

    def __len__(self):
        return len(self.models_a)

    def __getitem__(self, index):
        assert self.winners[index] in ["model_a", "model_b"], self.winners[index]
        if self.winners[index] == "model_a":
            return self.models_a[index], self.models_b[index], self.prompt_id[index]
        else:
            return self.models_b[index], self.models_a[index], self.prompt_id[index]


class MFModel_Train(nn.Module):
    def __init__(self, dim, num_models, num_prompts, text_dim, npy_path):
        super().__init__()
        self.P = nn.Embedding(num_models, dim)
        # Frozen prompt-embedding table (loaded from our MiniLM .npy). Not kept in the checkpoint.
        self.Q = nn.Embedding(num_prompts, text_dim).requires_grad_(False)
        embs = np.load(npy_path)
        if embs.shape[0] != num_prompts:
            raise ValueError(f"embeddings rows {embs.shape[0]} != num_prompts {num_prompts}")
        self.Q.weight.data.copy_(torch.tensor(embs, dtype=torch.float32))
        self.text_proj = nn.Linear(text_dim, dim, bias=False)
        self.classifier = nn.Linear(dim, 1, bias=False)  # bias must be False (matches original)

    def get_device(self):
        return self.P.weight.device

    def forward(self, model_win, model_loss, prompt, test=False, alpha=ALPHA):
        dev = self.get_device()
        model_win_embed = F.normalize(self.P(model_win.to(dev)), p=2, dim=1)
        model_loss_embed = F.normalize(self.P(model_loss.to(dev)), p=2, dim=1)
        prompt_embed = self.Q(prompt.to(dev))
        if not test:
            prompt_embed = prompt_embed + torch.randn_like(prompt_embed) * alpha
        prompt_embed = self.text_proj(prompt_embed)
        return self.classifier((model_win_embed - model_loss_embed) * prompt_embed).squeeze()

    @torch.no_grad()
    def predict(self, model_win, model_loss, prompt):
        return self.forward(model_win, model_loss, prompt, test=True) > 0


def train_loops(net, train_iter, test_iter, device, lr=LR, weight_decay=WEIGHT_DECAY,
                num_epochs=NUM_EPOCHS, patience=8):
    opt = Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    bce = nn.BCEWithLogitsLoss(reduction="mean")
    best_acc, best_state, patience_left = -1, None, patience

    def eval_acc():
        net.eval(); correct = total = 0
        with torch.no_grad():
            for a, b, p in test_iter:
                correct += (net.predict(a, b, p) == torch.ones_like(net(a, b, p))).sum().item()
                total += len(a)
        net.train()
        return correct / total

    for epoch in range(1, num_epochs + 1):
        net.train(); running = 0.0; nb = 0
        for a, b, p in train_iter:
            a, b, p = a.to(device), b.to(device), p.to(device)
            out = net(a, b, p)
            ls = bce(out, torch.ones_like(out))
            opt.zero_grad(); ls.backward(); opt.step()
            running += ls.item() * len(a); nb += len(a)
        acc = eval_acc()
        print(f"epoch {epoch:3d}  train_loss={running/nb:.4f}  val_acc={acc:.4f}")
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items() if k != "Q.weight"}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                print("Early stopping.")
                break
    return best_state, best_acc


# ---- split sorted by prompt idx so the val set spans the embedding range ----
data_shuffled = data.copy(); random.shuffle(data_shuffled)
n = len(data_shuffled); split = int(n * 0.95)
train_data, test_data = data_shuffled[:split], data_shuffled[split:]

num_prompts = len(emb)
net = MFModel_Train(DIM, NUM_MODELS, num_prompts, TEXT_DIM, os.path.join(OUT_DIR, "prompt_embeddings.npy"))
net = net.to(device)

train_iter = DataLoader(PairwiseDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
test_iter = DataLoader(PairwiseDataset(test_data), batch_size=1024, shuffle=False)

print(f"train={len(train_data)}  test={len(test_data)}  trainable_params={sum(p.numel() for p in net.parameters() if p.requires_grad)}")
best_state, best_acc = train_loops(net, train_iter, test_iter, device)
print(f"Best val acc: {best_acc:.4f}")

## 6. Save an inference-ready checkpoint (loadable by `MFModel.from_pretrained`)

The original loads via `MFModel.from_pretrained(...)` which is a Hugging Face `PyTorchModelHubMixin`, expecting a `pytorch_model.bin` (weights of `P`, `text_proj`, `classifier`) alongside a `config.json`. We save exactly that into `colab_outputs/checkpoint/`. **`Q` is excluded** — at inference the prompt embedding is computed fresh by the embedder (MiniLM in our local setup), not looked up.

In [ ]:
import json as _json

SAVE_DIR = os.path.join(OUT_DIR, "checkpoint")
os.makedirs(SAVE_DIR, exist_ok=True)

# Load the best state into a fresh inference-mode MFModel-style module (P, text_proj, classifier only)
class MFModelInference(nn.Module):
    def __init__(self, dim, num_models, text_dim):
        super().__init__()
        self.P = nn.Embedding(num_models, dim)
        self.use_proj = True
        self.text_proj = nn.Sequential(nn.Linear(text_dim, dim, bias=False))
        self.classifier = nn.Sequential(nn.Linear(dim, 1, bias=False))
    def forward(self, model_id, prompt_emb):
        model_embed = F.normalize(self.P(model_id), p=2, dim=1)
        prompt_embed = self.text_proj(prompt_emb)
        return self.classifier(model_embed * prompt_embed).squeeze()

inf_net = MFModelInference(DIM, NUM_MODELS, TEXT_DIM)
# The released MFModel wraps text_proj/classifier in nn.Sequential, so its state-dict keys are
# text_proj.0.weight / classifier.0.weight. The training net uses bare nn.Linear (text_proj.weight).
# Rename best_state's keys to the Sequential form so local MFModel.from_pretrained(...) loads exactly.
def _to_seq_keys(sd):
    out = {}
    for k, v in sd.items():
        if k == 'text_proj.weight':
            out['text_proj.0.weight'] = v
        elif k == 'classifier.weight':
            out['classifier.0.weight'] = v
        else:
            out[k] = v
    return out

seq_sd = _to_seq_keys(best_state)
torch.save(seq_sd, os.path.join(SAVE_DIR, 'pytorch_model.bin'))
print('Saved state-dict keys:', sorted(seq_sd.keys()))
_json.dump({
    "_name_or_path": "local-minilm-mf",
    "dim": DIM,
    "num_models": NUM_MODELS,
    "text_dim": TEXT_DIM,
    "num_classes": 1,
    "use_proj": True,
    "embedder": EMBEDDER,
}, open(os.path.join(SAVE_DIR, "config.json"), "w"))

# Also drop a human-readable note
_json.dump({"embedder": EMBEDDER, "text_dim": TEXT_DIM, "dim": DIM,
           "num_prompts": num_prompts, "best_val_acc": best_acc,
           "num_train": len(train_data), "num_test": len(test_data)},
          open(os.path.join(OUT_DIR, "meta.json"), "w"), indent=2)
print("Saved checkpoint to", SAVE_DIR)
print("Files in colab_outputs:", sorted(os.listdir(OUT_DIR)))

## 7. Zip everything for a single download

Download `colab_outputs.zip` once. Contents:
- `colab_outputs/checkpoint/pytorch_model.bin` + `config.json` — the trained router weights (load with `MFModel.from_pretrained(...)`).
- `colab_outputs/prompt_embeddings.npy` — cached MiniLM embeddings (only needed if you want to retrain/inspect; not needed for inference).
- `colab_outputs/pairwise_data.json` — the filtered pairwise battle data.
- `colab_outputs/meta.json` — training summary (embedder, dims, best val acc).

In [ ]:
import zipfile
zip_path = "/content/colab_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        for f in files:
            full = os.path.join(root, f)
            zf.write(full, os.path.relpath(full, "/content"))
print("Zip size (MB):", round(os.path.getsize(zip_path) / 1e6, 1))

from google.colab import files
files.download(zip_path)
print("Download started. Keep colab_outputs.zip safe — you only need this once.")

## 8. What to do locally after downloading

The notebook already produced everything that needed compute. On your machine you now need to make the **local** `routellm` use MiniLM at inference instead of OpenAI. The only change is in `routellm/routers/matrix_factorization/model.py`:

1. Load the embedder once: `embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=...)`.
2. In `MFModel.forward`, replace the OpenAI call:

   ```python
   # OLD (needs OpenAI API key):
   prompt_embed = OPENAI_CLIENT.embeddings.create(input=[prompt], model=self.embedding_model).data[0].embedding

   # NEW (local MiniLM):
   prompt_embed = embedder.encode([prompt], convert_to_numpy=True)[0]
   ```

3. Point `MatrixFactorizationRouter` at the downloaded checkpoint: `checkpoint_path` = local path to `colab_outputs/checkpoint`, and pass `text_dim=384` to the router so `MFModel.from_pretrained` builds the right `text_proj`.
4. Recalibrate the threshold (the winrate distribution differs from the OpenAI-trained checkpoint) — use `routellm/calibrate_threshold.py` on your strong-model fraction.

The factorization logic, model-pair routing, and the rest are unchanged — only the prompt-embedding source differs.